In [ ]:
import os
from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN")
login(hf_token)

if hf_token:
    login(hf_token)
    print("Successfully logged in to Hugging Face!")
else:
    print("Hugging Face token not found. Add it in Colab Secrets.")

Hugging Face token not found. Add it in Colab Secrets.


In [ ]:
# ============================================================
# Cell 2 — Evaluation Dependencies
# ============================================================

import sys

PYTHON = sys.executable

# Core dependencies
!{PYTHON} -m pip install -q numpy==1.19.5
!{PYTHON} -m pip install -q accelerate rouge-score bert-score nltk pyemd

# MoverScore directly from the AIPHES repository
!{PYTHON} -m pip install -q --force-reinstall \
    git+https://github.com/AIPHES/emnlp19-moverscore.git

# Remove torchao if present
!{PYTHON} -m pip uninstall torchao -y -q

# Compatible datasets version
!{PYTHON} -m pip install -q "datasets<4.0.0"

# Verify
try:
    from moverscore_v2 import get_idf_dict, word_mover_score
    print("✓ MoverScore installed successfully")
except Exception as e:
    print(f"✗ MoverScore import failed: {e}")
    print(f"Python: {PYTHON}")

ERROR: Exception:
Traceback (most recent call last):
  File "/venv/main/lib/python3.12/site-packages/pip/_internal/cli/base_command.py", line 109, in _run_wrapper
    status = _inner_run()
             ^^^^^^^^^^^^
  File "/venv/main/lib/python3.12/site-packages/pip/_internal/cli/base_command.py", line 102, in _inner_run
    return self.run(options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/venv/main/lib/python3.12/site-packages/pip/_internal/cli/req_command.py", line 106, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/venv/main/lib/python3.12/site-packages/pip/_internal/commands/install.py", line 478, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/venv/main/lib/python3.12/site-packages/pip/_internal/resolution/resolvelib/resolver.py", line 103, in resolve
    result = self._result = resolver.resolve(
                            ^^^^^^^^^^^^^^^^^
  File "/venv/main/lib/python3.12/sit

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ MoverScore installed successfully


In [ ]:
!{sys.executable} -m pip install --upgrade pip setuptools wheel
!{sys.executable} -m pip install numpy scikit-learn
!{sys.executable} -m pip install accelerate rouge-score bert-score nltk torch torchvision pyemd datasets transformers peft "torchao>=0.16.0"
import nltk; nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True)

import torch
import gc
import math
import os
import threading
import time
import json
from typing import Optional
from PIL import Image

# Patch for numpy.float deprecation in MoverScore
import numpy as np
if not hasattr(np, 'float'):
    np.float = float

import re
import numpy as np
from transformers import EvalPrediction
import nltk
nltk.download('punkt', quiet=True)

from datasets import load_dataset
from transformers import (
    Gemma3ForConditionalGeneration,
    AutoProcessor,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainerCallback,
)
from moverscore_v2 import get_idf_dict, word_mover_score
from peft import LoraConfig, get_peft_model


def load_image_from_source(image_source):
    """
    Robust image loading for VQAv2 examples.

    Unlike VQAonline, HuggingFaceM4/VQAv2's `image` column is already a
    decoded PIL.Image (the loading script embeds the COCO images directly),
    so this is mostly a defensive convert-to-RGB helper rather than a
    downloader/cache like VQAonline needed.
    """
    try:
        if isinstance(image_source, Image.Image):
            return image_source.convert('RGB') if image_source.mode != 'RGB' else image_source

        # Fallback in case a path/bytes slips through instead of a PIL Image.
        if isinstance(image_source, str) and os.path.exists(image_source):
            image = Image.open(image_source)
            return image.convert('RGB') if image.mode != 'RGB' else image

        if isinstance(image_source, bytes):
            from io import BytesIO
            image = Image.open(BytesIO(image_source))
            return image.convert('RGB') if image.mode != 'RGB' else image

        return None

    except Exception as e:
        print(f"Failed to load image: {e}")
        return None


def extract_raw_answers(example):
    """
    Return the list of raw human answer strings for a VQAv2 example, for
    use with the official VQA soft-accuracy metric.

    HuggingFaceM4/VQAv2's `answers` field is a list of dicts like
    {"answer": "net", "answer_confidence": "yes", "answer_id": 1}. This
    also tolerates a flat list-of-strings format in case the dataset
    source is ever swapped for a different VQAv2 mirror.
    """
    answers_field = example.get('answers')
    if not answers_field:
        return []
    raw = []
    for a in answers_field:
        ans = a.get('answer') if isinstance(a, dict) else a
        if ans:
            raw.append(str(ans))
    return raw



class ReproducibilityTracker:
    """
    Tracks training reproducibility / efficiency metrics.

    Step latency is measured from on_step_begin to on_step_end,
    corresponding to one optimizer step including gradient accumulation.
    """

    def __init__(self):
        self.step_times = []
        self.training_start_time = None
        self.training_end_time = None
        self._step_start_time = None

        self.peak_memory_allocated = 0.0
        self.peak_memory_reserved = 0.0

    def start(self):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()

        self.training_start_time = time.perf_counter()

    def start_step(self):
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        self._step_start_time = time.perf_counter()

    def end_step(self):
        if self._step_start_time is None:
            return

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - self._step_start_time

        if elapsed > 0:
            self.step_times.append(elapsed)

        self._step_start_time = None

    def stop(self):
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        self.training_end_time = time.perf_counter()

        if torch.cuda.is_available():
            self.peak_memory_allocated = (
                torch.cuda.max_memory_allocated() / (1024 ** 3)
            )
            self.peak_memory_reserved = (
                torch.cuda.max_memory_reserved() / (1024 ** 3)
            )

    def summary(self):
        if (
            self.training_start_time is None
            or self.training_end_time is None
        ):
            return {}

        total_time = (
            self.training_end_time
            - self.training_start_time
        )

        if self.step_times:
            # Exclude first step because CUDA/kernel initialization
            # can make it substantially slower.
            measured_steps = (
                self.step_times[1:]
                if len(self.step_times) > 1
                else self.step_times
            )

            mean_ms = float(np.mean(measured_steps)) * 1000
            p99_ms = float(
                np.percentile(measured_steps, 99)
            ) * 1000
        else:
            mean_ms = 0.0
            p99_ms = 0.0

        return {
            "training_time_sec": total_time,
            "training_time_min": total_time / 60,
            "training_time_hours": total_time / 3600,

            "num_optimizer_steps": len(self.step_times),

            "mean_step_ms": mean_ms,
            "p99_step_ms": p99_ms,

            "peak_gpu_memory_allocated_gb":
                self.peak_memory_allocated,

            "peak_gpu_memory_reserved_gb":
                self.peak_memory_reserved,
        }


class ReproducibilityCallback(TrainerCallback):

    def __init__(self, tracker):
        self.tracker = tracker

    def on_train_begin(self, args, state, control, **kwargs):
        self.tracker.start()

    def on_step_begin(self, args, state, control, **kwargs):
        self.tracker.start_step()

    def on_step_end(self, args, state, control, **kwargs):
        self.tracker.end_step()

    def on_train_end(self, args, state, control, **kwargs):
        self.tracker.stop()

class LoRAConfig:
    """Configuration for LoRA-only training on Gemma 3-4B-IT models."""

    def __init__(self):
        # Gemma 3-4B-IT model ID
        self.model_id = "google/gemma-3-4b-it"
        self.batch_size = 1
        self.gradient_accumulation_steps = 1
        self.num_epochs = 3
        self.learning_rate = 1e-5
        self.precision = "bf16"
        self.max_length = 1024

        # LoRA configuration
        self.lora_rank = 4
        self.lora_alpha = 16
        self.lora_dropout = 0.05

        # Gemma 3 target modules
        self.lora_target_modules = [
            "q_proj",
            "k_proj",
            "v_proj"
        ]

        # Vision encoder LoRA
        self.apply_lora_to_vision = True
        self.vision_lora_modules = [
            "q_proj",
            "k_proj",
            "v_proj"
        ]

        # Dataset configuration
        # HuggingFaceM4/VQAv2 uses a legacy loading script (requires
        # trust_remote_code=True) and ships train/validation/test/testdev.
        # Only train + validation have public answers; test/testdev are the
        # blind challenge-server splits with no ground truth.
        self.dataset_name = "HuggingFaceM4/VQAv2"
        self.num_workers = 0
        self.pin_memory = True
        self.drop_last = True

        self.answer_max_length = 512
        self.ignore_index = -100

        # Since VQAv2's real "test" split has no answers, we hold out a
        # slice of "validation" to use as our own answer-bearing test set.
        self.held_out_test_size = 1000


def main():
    config = LoRAConfig()

    print("Loading processor and model for Gemma 3-4B-IT...")
    processor = AutoProcessor.from_pretrained(
        config.model_id,
        token=hf_token,
        trust_remote_code=True,
        padding_side="left",
    )

    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    model = Gemma3ForConditionalGeneration.from_pretrained(
        config.model_id,
        device_map="auto",
        token=hf_token,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )

    # Apply LoRA configuration
    target_modules = config.lora_target_modules.copy()

    lora_config = LoraConfig(
        r=config.lora_rank,
        lora_alpha=config.lora_alpha,
        target_modules=target_modules,
        lora_dropout=config.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        inference_mode=False,
        use_rslora=True,
    )

    model = get_peft_model(model, lora_config)

    # Count vision and language modules
    from peft.tuners.lora import LoraLayer
    vision_modules = 0
    language_modules = 0

    for name, module in model.named_modules():
        if isinstance(module, LoraLayer) and module.r.get('default', 0) > 0:
            name_lower = name.lower()
            is_vision = any([
                'vision_tower' in name_lower,
                'vision_model' in name_lower,
                'vision_encoder' in name_lower,
                'visual' in name_lower and 'language' not in name_lower,
                name.startswith('model.vision'),
                name.startswith('vision_tower'),
            ])

            if is_vision:
                vision_modules += 1
            else:
                language_modules += 1

    print(f"Instrumented {vision_modules} vision modules + {language_modules} language modules")
    print(f"Total LoRA modules: {vision_modules + language_modules}")

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    print(f"Trainable params: {trainable_params:,}")
    print(f"Total params: {total_params:,}")
    print(f"Percentage: {100 * trainable_params / total_params:.3f}%")

    model.enable_input_require_grads()
    model.config.use_cache = False

    print("Loading VQAv2 dataset with splits...")
    # trust_remote_code=True is required: HuggingFaceM4/VQAv2 ships a
    # dataset loading script rather than pre-converted parquet files.
    # full_dataset = load_dataset(config.dataset_name, trust_remote_code=True)

    # train_ds = full_dataset["train"]
    # val_full_ds = full_dataset["validation"]
    # # NOTE: full_dataset["test"] / "testdev" exist but carry no answers
    # # (they're the blind VQA challenge splits), so we don't use them for
    # # supervised training or metric-based eval here.

    # train_dataset = train_ds.shuffle(seed=42).select(range(32))

    # # Carve the validation split into a val set and a held-out "test" set
    # # (both have real answers, unlike the official test split).
    # val_shuffled = val_full_ds.shuffle(seed=42)
    # val_dataset = val_shuffled.select(range(4))
    # test_dataset = val_shuffled.select(range(4, 4 + config.held_out_test_size))

    from datasets import load_dataset, Dataset

    print("Streaming VQAv2 dataset (no full download)...")
    streamed = load_dataset(config.dataset_name, trust_remote_code=True, streaming=True)

    # Shuffle with a buffer (approximate shuffle, not a full-dataset shuffle,
    # but fine for pulling a small subset) then take only what we need.
    train_stream = streamed["train"].shuffle(seed=42, buffer_size=1000)
    train_examples = list(train_stream.take(4000))

    val_stream = streamed["validation"].shuffle(seed=42, buffer_size=1000)
    val_examples = list(val_stream.take(1000 + config.held_out_test_size))  # val + held-out test

    val_dataset = Dataset.from_list(val_examples[:1000])
    test_dataset = Dataset.from_list(val_examples[1000:])
    train_dataset = Dataset.from_list(train_examples)

    print(f"Final splits → Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

    def tokenize_vqa(example):
        """Process VQAv2 examples with robust image loading for Gemma 3."""
        try:
            if 'image' not in example or example['image'] is None:
                return None

            image = load_image_from_source(example['image'])

            if image is None:
                return None

            if not isinstance(image, Image.Image):
                return None
            if image.mode != 'RGB':
                image = image.convert('RGB')

            question = (example.get('question') or '').strip()
            # VQAv2 provides `multiple_choice_answer` (the majority/consensus
            # answer) as well as a 10-annotator `answers` list. We train
            # against the consensus answer, analogous to VQAonline's
            # single `answer` field.
            answer = (example.get('multiple_choice_answer') or '').strip()

            if not question or not answer:
                return None

            # VQAv2 has no `context` field (unlike VQAonline), so the
            # prompt is just the question.
            text_content = f"Question: {question}\n\nProvide a concise answer."

            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": text_content}
                    ]
                }
            ]

            text_prompt = processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            inputs = processor(
                text=[text_prompt],
                images=[image],
                return_tensors="pt",
                padding="max_length",
                max_length=config.max_length,
                truncation=True,
            )

            answer_tokens = processor.tokenizer(
                answer,
                add_special_tokens=False,
                return_tensors="pt"
            )['input_ids'].squeeze(0)

            input_ids = inputs['input_ids'].squeeze(0)
            attention_mask = inputs['attention_mask'].squeeze(0)

            token_type_ids = inputs.get('token_type_ids', None)
            if token_type_ids is not None:
                token_type_ids = token_type_ids.squeeze(0)

            if not isinstance(input_ids, torch.Tensor):
                input_ids = torch.tensor(input_ids, dtype=torch.long)
            if not isinstance(attention_mask, torch.Tensor):
                attention_mask = torch.tensor(attention_mask, dtype=torch.long)
            if not isinstance(answer_tokens, torch.Tensor):
                answer_tokens = torch.tensor(answer_tokens, dtype=torch.long)
            if token_type_ids is not None and not isinstance(token_type_ids, torch.Tensor):
                token_type_ids = torch.tensor(token_type_ids, dtype=torch.long)

            labels = input_ids.clone()
            labels[:] = config.ignore_index

            answer_len = min(len(answer_tokens), len(labels))
            if answer_len > 0:
                labels[-answer_len:] = answer_tokens[:answer_len]

            output = {
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': labels,
                # Raw (un-normalized) list of the ~10 human answers, kept
                # around purely for the official VQA soft-accuracy metric
                # at eval time. Not used in training/loss.
                'vqa_answers': extract_raw_answers(example),
            }
            if token_type_ids is not None:
                output['token_type_ids'] = token_type_ids

            if 'pixel_values' in inputs:
                pixel_values = inputs['pixel_values']

                if isinstance(pixel_values, torch.Tensor):
                    if pixel_values.dim() >= 2 and pixel_values.shape[0] == 1:
                        pixel_values = pixel_values.squeeze(0)
                    output['pixel_values'] = pixel_values
                elif isinstance(pixel_values, list) and len(pixel_values) > 0:
                    if isinstance(pixel_values[0], torch.Tensor):
                        output['pixel_values'] = pixel_values[0] if len(pixel_values) == 1 else torch.cat(pixel_values, dim=0)

            for key in ['input_ids', 'attention_mask', 'labels', 'token_type_ids']:
                if key in output and not isinstance(output[key], torch.Tensor):
                    print(f"ERROR: {key} is not a tensor: {type(output[key])}")
                    return None

            return output

        except Exception as e:
            print(f"Error processing example: {e}")
            import traceback
            traceback.print_exc()
            return None

    print("Tokenizing datasets...")
    print("Processing training data...")
    tokenized_train_dataset = train_dataset.map(
        tokenize_vqa,
        remove_columns=train_dataset.column_names,
        num_proc=1,
        desc="Processing training data",
        load_from_cache_file=False
    )

    print("Filtering valid training samples...")
    tokenized_train_dataset = tokenized_train_dataset.filter(
        lambda x: x is not None and 'input_ids' in x and 'labels' in x,
        desc="Filtering training data"
    )

    print("Processing validation data...")
    tokenized_val_dataset = val_dataset.map(
        tokenize_vqa,
        remove_columns=val_dataset.column_names,
        num_proc=1,
        desc="Processing validation data",
        load_from_cache_file=False
    )

    print("Filtering valid validation samples...")
    tokenized_val_dataset = tokenized_val_dataset.filter(
        lambda x: x is not None and 'input_ids' in x and 'labels' in x,
        desc="Filtering validation data"
    )

    print("Processing test data (held out from validation)...")
    tokenized_test_dataset = test_dataset.map(
        tokenize_vqa,
        remove_columns=test_dataset.column_names,
        num_proc=1,
        desc="Tokenizing test",
        load_from_cache_file=False,
    )

    print("Filtering valid test samples...")
    tokenized_test_dataset = tokenized_test_dataset.filter(
        lambda x: x is not None and 'input_ids' in x and 'labels' in x,
        desc="Filtering test data"
    )

    train_len = len(tokenized_train_dataset)
    val_len = len(tokenized_val_dataset)
    test_len = len(tokenized_test_dataset)

    if train_len == 0:
        raise ValueError(
            "No training samples after processing. "
            "Possible causes: images not loading correctly, dataset format incompatible, "
            "or all examples failed validation."
        )

    print(f"\nSuccessfully processed datasets:")
    print(f"Training samples: {train_len}")
    print(f"Validation samples: {val_len}")
    print(f"Test samples: {test_len}")

    eff_batch = config.batch_size * config.gradient_accumulation_steps
    total_steps = math.ceil(train_len / eff_batch) * config.num_epochs

    print(f"\nTraining Configuration:")
    print(f"Total training steps: {total_steps}")
    print(f"Effective batch size: {eff_batch}")
    print(f"Steps per epoch: {math.ceil(train_len / eff_batch)}")

    training_args = Seq2SeqTrainingArguments(
        output_dir="./results_vqa_lora_only_gemma3",
        per_device_train_batch_size=config.batch_size,
        per_device_eval_batch_size=config.batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        num_train_epochs=config.num_epochs,
        learning_rate=config.learning_rate,
        bf16=(config.precision == "bf16"),
        fp16=(config.precision == "fp16"),
        max_grad_norm=1.5,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        dataloader_num_workers=config.num_workers,
        dataloader_pin_memory=config.pin_memory,
        dataloader_drop_last=config.drop_last,
        remove_unused_columns=False,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_steps=50,
        warmup_steps=100,
        optim="adamw_torch",
        #metric_for_best_model="loss",
        prediction_loss_only=False,  # Need full predictions for metrics
        predict_with_generate=True,  # Use generation for evaluation
        generation_max_length=config.max_length,
    )

    def collate_fn(batch):
        """Custom collator for Gemma 3 inputs with proper batching."""
        batch = [b for b in batch if b is not None and isinstance(b, dict) and 'input_ids' in b]

        if len(batch) == 0:
            print("WARNING: Empty batch after filtering - creating dummy batch")
            dummy_input_ids = torch.zeros((1, config.max_length), dtype=torch.long)
            dummy_attention_mask = torch.zeros((1, config.max_length), dtype=torch.long)
            dummy_labels = torch.full((1, config.max_length), config.ignore_index, dtype=torch.long)
            dummy_token_type_ids = torch.zeros((1, config.max_length), dtype=torch.long)

            return {
                'input_ids': dummy_input_ids,
                'attention_mask': dummy_attention_mask,
                'labels': dummy_labels,
                'token_type_ids': dummy_token_type_ids,
            }

        try:
            for i, item in enumerate(batch):
                for key in ['input_ids', 'attention_mask', 'labels', 'token_type_ids']:
                    if key in item and not isinstance(item[key], torch.Tensor):
                        batch[i][key] = torch.tensor(item[key], dtype=torch.long)

            input_ids = torch.nn.utils.rnn.pad_sequence(
                [b['input_ids'] for b in batch],
                batch_first=True,
                padding_value=processor.tokenizer.pad_token_id
            )

            attention_mask = torch.nn.utils.rnn.pad_sequence(
                [b['attention_mask'] for b in batch],
                batch_first=True,
                padding_value=0
            )

            labels = torch.nn.utils.rnn.pad_sequence(
                [b['labels'] for b in batch],
                batch_first=True,
                padding_value=config.ignore_index
            )

            output = {
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': labels,
            }

            if 'token_type_ids' in batch[0]:
                token_type_ids_list = [
                    b['token_type_ids'] for b in batch
                    if 'token_type_ids' in b and isinstance(b['token_type_ids'], torch.Tensor)
                ]
                if token_type_ids_list:
                    token_type_ids = torch.nn.utils.rnn.pad_sequence(
                        token_type_ids_list,
                        batch_first=True,
                        padding_value=0
                    )
                    output['token_type_ids'] = token_type_ids

            if 'pixel_values' in batch[0] and batch[0]['pixel_values'] is not None:
                pixel_values_list = []

                for b in batch:
                    pv = b['pixel_values']

                    if isinstance(pv, list):
                        if len(pv) > 0 and isinstance(pv[0], torch.Tensor):
                            pv = torch.stack(pv) if len(pv) > 1 else pv[0]
                        elif len(pv) > 0 and isinstance(pv[0], list):
                            pv = torch.tensor(pv)
                        else:
                            continue

                    if isinstance(pv, torch.Tensor):
                        while pv.dim() > 0 and pv.shape[0] == 1:
                            pv = pv.squeeze(0)
                        pixel_values_list.append(pv)

                if len(pixel_values_list) > 0:
                    try:
                        output['pixel_values'] = torch.stack(pixel_values_list, dim=0)
                    except RuntimeError:
                        max_shapes = [max(pv.shape[i] for pv in pixel_values_list) for i in range(pixel_values_list[0].dim())]
                        padded_pvs = []
                        for pv in pixel_values_list:
                            if list(pv.shape) != max_shapes:
                                padding = []
                                for i in range(len(max_shapes) - 1, -1, -1):
                                    padding.extend([0, max_shapes[i] - pv.shape[i]])
                                pv = torch.nn.functional.pad(pv, padding)
                            padded_pvs.append(pv)
                        output['pixel_values'] = torch.stack(padded_pvs, dim=0)

            # Pass the raw per-example human answers through untouched
            # (list of lists of strings) — needed for the official VQA
            # soft-accuracy metric. This is popped out again in
            # LoRATrainer.prediction_step before the tensors go to the model.
            output['vqa_answers'] = [b.get('vqa_answers', []) for b in batch]

            return output

        except Exception as e:
            print(f"ERROR in collate_fn: {e}")
            import traceback
            traceback.print_exc()

            dummy_input_ids = torch.zeros((1, config.max_length), dtype=torch.long)
            dummy_attention_mask = torch.zeros((1, config.max_length), dtype=torch.long)
            dummy_labels = torch.full((1, config.max_length), config.ignore_index, dtype=torch.long)
            dummy_token_type_ids = torch.zeros((1, config.max_length), dtype=torch.long)

            return {
                'input_ids': dummy_input_ids,
                'attention_mask': dummy_attention_mask,
                'labels': dummy_labels,
                'token_type_ids': dummy_token_type_ids,
            }

    def make_compute_metrics(processor, eval_answers_buffer):
        import re
        import numpy as np
        from rouge_score import rouge_scorer
        from nltk.translate.meteor_score import meteor_score
        from bert_score import score as bert_score
        from collections import defaultdict
        import nltk
        nltk.download('wordnet', quiet=True)
        nltk.download('omw-1.4', quiet=True)

        # PATCH FOR MOVERSCORE COMPATIBILITY
        import transformers.tokenization_utils_base as tub
        if not hasattr(tub.PreTrainedTokenizerBase, '_moverscore_patched'):
            original_init = tub.PreTrainedTokenizerBase.__init__
            def patched_init(self, *args, **kwargs):
                original_init(self, *args, **kwargs)
                if not hasattr(self, 'max_len'):
                    self.max_len = self.model_max_length
            tub.PreTrainedTokenizerBase.__init__ = patched_init
            tub.PreTrainedTokenizerBase._moverscore_patched = True
            print("✓ Applied moverscore compatibility patch")

        # Now import moverscore
        moverscore_available = False
        try:
            from moverscore_v2 import get_idf_dict, word_mover_score
            moverscore_available = True
            print("✓ MoverScore loaded successfully")
        except ImportError as e:
            print(f"✗ MoverScore not available: {e}")

        scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

        def extract_answer(text):
            """Clean Gemma output safely without truncating content."""

            if text is None:
                return ""

            # Remove special tokens
            text = text.replace("<bos>", "")
            text = text.replace("<eos>", "")
            text = text.replace("<start_of_turn>", "")
            text = text.replace("<end_of_turn>", "")

            # Remove leading 'model' tag if present
            text = re.sub(r"^\s*model\s*", "", text, flags=re.IGNORECASE)

            # Normalize whitespace
            text = re.sub(r"\s+", " ", text).strip()

            return text

        # Official VQA eval answer processing (from the VQA project's
        # vqaEval.py), used so exact-match comparisons for accuracy (and
        # the text-similarity metrics below) aren't broken by trailing
        # punctuation, articles, or number-word vs. digit mismatches.
        _vqa_contractions = {
            "aint": "ain't", "arent": "aren't", "cant": "can't", "couldve": "could've",
            "couldnt": "couldn't", "didnt": "didn't", "doesnt": "doesn't", "dont": "don't",
            "hadnt": "hadn't", "hasnt": "hasn't", "havent": "haven't", "hed": "he'd",
            "hes": "he's", "im": "i'm", "ive": "i've", "isnt": "isn't", "its": "it's",
            "lets": "let's", "shant": "shan't", "shes": "she's", "shouldve": "should've",
            "shouldnt": "shouldn't", "thats": "that's", "theres": "there's", "theyd": "they'd",
            "theyre": "they're", "theyve": "they've", "wasnt": "wasn't", "wed": "we'd",
            "were": "we're", "weve": "we've", "werent": "weren't", "whats": "what's",
            "wheres": "where's", "whos": "who's", "wont": "won't", "wouldve": "would've",
            "wouldnt": "wouldn't", "youd": "you'd", "youre": "you're", "youve": "you've",
        }
        _vqa_number_words = {
            "none": "0", "zero": "0", "one": "1", "two": "2", "three": "3", "four": "4",
            "five": "5", "six": "6", "seven": "7", "eight": "8", "nine": "9", "ten": "10",
        }
        _vqa_articles = {"a", "an", "the"}
        _vqa_punct = [';', '/', '[', ']', '"', '{', '}', '(', ')', '=', '+', '\\',
                      '_', '-', '>', '<', '@', '`', ',', '?', '!', '.', ':']

        def normalize(s):
            """Official VQA-style answer normalization."""
            s = s.lower().strip()
            # Remove punctuation entirely (rather than spacing it out),
            # so "brown." normalizes to "brown", not "brown .".
            for p in _vqa_punct:
                s = s.replace(p, ' ')
            s = re.sub(r"\s+", " ", s).strip()

            words = []
            for word in s.split():
                word = _vqa_number_words.get(word, word)
                if word in _vqa_articles:
                    continue
                word = _vqa_contractions.get(word, word)
                words.append(word)
            return ' '.join(words).strip()

        def compute_vqa_metrics(p: EvalPrediction):
            pred_ids = np.asarray(p.predictions, dtype=np.int64)
            labels = np.asarray(p.label_ids, dtype=np.int64)

            vocab_size = processor.tokenizer.vocab_size
            pad_token_id = processor.tokenizer.pad_token_id

            pred_ids = np.clip(pred_ids, 0, vocab_size - 1)
            labels[labels < 0] = pad_token_id
            labels = np.clip(labels, 0, vocab_size - 1)

            try:
                pred_texts_raw = processor.batch_decode(pred_ids, skip_special_tokens=True)
                label_texts = processor.batch_decode(labels, skip_special_tokens=True)
            except Exception as e:
                print(f"[Metrics] Decode error: {e}")
                return {"rouge_l": 0.0, "meteor": 0.0, "bertscore_f1": 0.0, "moverscore": 0.0, "accuracy": 0.0}

            pred_texts = [extract_answer(t) for t in pred_texts_raw]
            pred_norm = [normalize(t) for t in pred_texts]
            label_norm = [normalize(t) for t in label_texts]

            for i in range(min(3, len(pred_norm))):
                print(f"\n[Sample {i+1}]")
                print(f"Generated: {pred_texts[i][:100]}")
                print(f"Reference: {label_texts[i][:100]}")

            valid_preds = []
            valid_labels = []
            for p, l in zip(pred_norm, label_norm):
                if p and l and len(p.split()) > 0 and len(l.split()) > 0:
                    valid_preds.append(p)
                    valid_labels.append(l)

            # ROUGE-L
            rouge_ls = []
            for p, l in zip(pred_norm, label_norm):
                if p and l:
                    rouge_ls.append(scorer.score(l, p)['rougeL'].fmeasure)
            rouge_l = np.mean(rouge_ls) if rouge_ls else 0.0

            # METEOR
            meteors = []
            for p, l in zip(pred_norm, label_norm):
                if p and l:
                    meteors.append(meteor_score([l.split()], p.split()))
            meteor = np.mean(meteors) if meteors else 0.0

            # BERTscore
            if valid_preds and valid_labels:
                try:
                    P, R, F1 = bert_score(
                        valid_preds,
                        valid_labels,
                        lang="en",
                        model_type="bert-base-uncased",
                        rescale_with_baseline=False
                    )
                    bertscore_f1 = F1.mean().item()
                except Exception as e:
                    print(f"[Metrics] BERTScore error: {e}")
                    bertscore_f1 = 0.0
            else:
                bertscore_f1 = 0.0

            # Accuracy — official VQA soft-accuracy formula:
            #   acc(pred) = average over each of the N human answers,
            #               leaving that one out, of
            #               min(#matches-among-the-other-(N-1) / 3, 1)
            # Requires the raw ~10 human answers per question, which
            # LoRATrainer.prediction_step buffers into eval_answers_buffer
            # (popped from each batch before it reaches model.generate).
            raw_answers_list = eval_answers_buffer.get("answers", [])

            print(f"\n[Accuracy Debug] buffered answer-sets: {len(raw_answers_list)} | predictions: {len(pred_norm)}")
            for i in range(min(3, len(raw_answers_list), len(pred_norm))):
                print(f"[Accuracy Debug] Sample {i+1}")
                print(f"  Prediction (normalized): {pred_norm[i]!r}")
                print(f"  Raw human answers       : {raw_answers_list[i]}")
                print(f"  Normalized human answers: {[normalize(a) for a in raw_answers_list[i] if a]}")

            if len(raw_answers_list) != len(pred_norm):
                print(
                    f"[Metrics] WARNING: buffered answers ({len(raw_answers_list)}) "
                    f"!= predictions ({len(pred_norm)}); falling back to "
                    f"normalized exact-match accuracy against the single "
                    f"training answer instead of the official VQA formula."
                )
                matches = [1.0 if p == l else 0.0 for p, l in zip(pred_norm, label_norm) if l]
                accuracy = float(np.mean(matches)) if matches else 0.0
            else:
                per_question_accs = []
                for pred, raw_answers in zip(pred_norm, raw_answers_list):
                    norm_answers = [normalize(a) for a in raw_answers if a]
                    n = len(norm_answers)
                    if n == 0:
                        continue
                    if n == 1:
                        # Can't do leave-one-out with a single annotator;
                        # fall back to the plain ratio-based formula.
                        matches = sum(1 for a in norm_answers if a == pred)
                        per_question_accs.append(min(matches / 3.0, 1.0))
                        continue
                    leave_one_out_accs = []
                    for i in range(n):
                        others = norm_answers[:i] + norm_answers[i + 1:]
                        matches = sum(1 for a in others if a == pred)
                        leave_one_out_accs.append(min(matches / 3.0, 1.0))
                    per_question_accs.append(float(np.mean(leave_one_out_accs)))
                accuracy = float(np.mean(per_question_accs)) if per_question_accs else 0.0

            # MoverScore
            moverscore_val = 0.0
            if moverscore_available and valid_preds and valid_labels:
                try:
                    from moverscore_v2 import get_idf_dict, word_mover_score

                    idf_dict_ref = get_idf_dict(valid_labels)
                    idf_dict_hyp = get_idf_dict(valid_preds)

                    scores = word_mover_score(
                        valid_labels,
                        valid_preds,
                        idf_dict_ref,
                        idf_dict_hyp,
                        stop_words=[],
                        n_gram=1,
                        remove_subwords=True
                    )

                    if scores and len(scores) > 0:
                        moverscore_val = float(np.mean(scores))
                        print(f"[MoverScore] Score: {moverscore_val:.4f}")

                except Exception as e:
                    print(f"[MoverScore] Error: {e}")
                    import traceback
                    traceback.print_exc()

            return {
                "rouge_l": float(rouge_l),
                "meteor": float(meteor),
                "bertscore_f1": float(bertscore_f1),
                "moverscore": float(moverscore_val),
                "accuracy": accuracy,
            }

        return compute_vqa_metrics

    # Custom Trainer with VRAM clearing before evaluation
    class LoRATrainer(Seq2SeqTrainer):
        """Seq2SeqTrainer subclass that clears VRAM before evaluation."""

        def __init__(self, processor, eval_answers_buffer, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.processor = processor
            # Mutable dict shared with make_compute_metrics(), used to pass
            # the raw per-question human answers (popped out of each batch
            # below) through to compute_metrics for the official VQA
            # soft-accuracy formula, since EvalPrediction only carries
            # decoded predictions/labels.
            self.eval_answers_buffer = eval_answers_buffer

        def evaluate(self, *args, **kwargs):
            print("\nClearing VRAM before evaluation...")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            self.eval_answers_buffer["answers"] = []
            return super().evaluate(*args, **kwargs)

        def prediction_step(
            self,
            model: torch.nn.Module,
            inputs: dict,
            prediction_loss_only: bool,
            ignore_keys: Optional[list] = None,
        ):
            """Memory-efficient prediction step from GRIT."""
            has_labels = "labels" in inputs
            if has_labels:
                labels = inputs.pop("labels")
            else:
                labels = None

            # Pull the raw human-answer lists out before anything touches
            # the model — they're plain Python lists, not tensors, and
            # model.generate()/forward() would choke on an unexpected kwarg.
            vqa_answers_batch = inputs.pop("vqa_answers", None)
            if vqa_answers_batch is not None:
                self.eval_answers_buffer["answers"].extend(vqa_answers_batch)

            input_ids = inputs["input_ids"]
            input_len = input_ids.shape[1]

            with torch.no_grad():
                if prediction_loss_only:
                    outputs = model(**inputs)
                    loss = outputs.loss if has_labels else None
                    return (loss, None, None)

                # Generate (full sequence)
                generated_tokens = model.generate(
                    **inputs,
                    max_new_tokens=512,
                    do_sample=False,
                    pad_token_id=self.processor.tokenizer.pad_token_id,
                    eos_token_id=self.processor.tokenizer.eos_token_id,
                )

                # Slice to generated part only (answer)
                generated_tokens = generated_tokens[:, input_len:]

                # Loss (full inputs for compute_loss)
                loss = None
                if has_labels:
                    inputs_copy = inputs.copy()
                    inputs_copy["labels"] = labels
                    loss = self.compute_loss(model, inputs_copy).detach()

            # Labels processing
            if labels is not None:
                if labels.dim() == 3:
                    labels = torch.argmax(labels, dim=-1)
                elif labels.dim() == 1:
                    batch_size = input_ids.shape[0]
                    labels = labels.view(batch_size, -1)

            return loss, generated_tokens, labels

    # Shared mutable buffer: LoRATrainer.prediction_step fills it with the
    # raw per-question human answers for each eval batch; compute_metrics
    # reads it to compute the official VQA soft-accuracy formula.
    eval_answers_buffer = {"answers": []}

    tracker = ReproducibilityTracker()

    trainer = LoRATrainer(
        processor=processor,
        eval_answers_buffer=eval_answers_buffer,
        model=model,
        args=training_args,
        train_dataset=tokenized_train_dataset,
        eval_dataset=tokenized_val_dataset,
        data_collator=collate_fn,
        compute_metrics=make_compute_metrics(processor, eval_answers_buffer),
        callbacks=[
            ReproducibilityCallback(tracker)
        ],
    )

    print("\n" + "="*80)
    print("STARTING LORA-ONLY TRAINING FOR GEMMA 3-4B-IT MODEL (ABLATION STUDY)")
    print("="*80)
    print(f"Model: {config.model_id}")
    print(f"Dataset: {config.dataset_name}")
    print(f"LoRA rank: {config.lora_rank}")
    print(f"Batch size: {config.batch_size} x {config.gradient_accumulation_steps} = {eff_batch}")
    print(f"Learning rate: {config.learning_rate}")
    print(f"Precision: {config.precision}")
    print(f"Training samples: {train_len}")
    print(f"Validation samples: {val_len}")
    print(f"Test samples: {test_len}")
    print("="*80 + "\n")

    try:
        print("Beginning training...")
        trainer.train()

        # ============================================================
        # FINAL TEST EVALUATION
        # ============================================================
        print("\n" + "="*80)
        print("FINAL TEST EVALUATION")
        print("="*80)

        # Run test evaluation FIRST
        test_results = trainer.evaluate(
            eval_dataset=tokenized_test_dataset
        )

        # ============================================================
        # REPRODUCIBILITY METRICS
        # ============================================================
        repro_metrics = tracker.summary()
        total_steps = trainer.state.global_step

        # ============================================================
        # PRINT TEST RESULTS
        # ============================================================
        print(f"Test Loss          : {test_results['eval_loss']:.5f}")
        print(f"Test ROUGE-L       : {test_results['eval_rouge_l']:.4f}")
        print(f"Test METEOR         : {test_results['eval_meteor']:.4f}")
        print(f"Test BERTscore F1   : {test_results['eval_bertscore_f1']:.4f}")
        print(f"Test MoverScore     : {test_results['eval_moverscore']:.4f}")
        print(f"Test Accuracy       : {test_results['eval_accuracy']:.4f}")

        # ============================================================
        # PRINT REPRODUCIBILITY METRICS
        # ============================================================
        print("\n" + "="*80)
        print("TRAINING REPRODUCIBILITY METRICS")
        print("="*80)

        print(
            f"Training wall-clock time : "
            f"{repro_metrics['training_time_hours']:.4f} h"
        )

        print(
            f"Training wall-clock time : "
            f"{repro_metrics['training_time_min']:.2f} min"
        )

        print(
            f"Optimizer steps          : "
            f"{total_steps}"
        )

        print(
            f"Mean step time           : "
            f"{repro_metrics['mean_step_ms']:.2f} ms"
        )

        print(
            f"P99 step time            : "
            f"{repro_metrics['p99_step_ms']:.2f} ms"
        )

        print(
            f"Peak GPU memory allocated: "
            f"{repro_metrics['peak_gpu_memory_allocated_gb']:.2f} GB"
        )

        print(
            f"Peak GPU memory reserved : "
            f"{repro_metrics['peak_gpu_memory_reserved_gb']:.2f} GB"
        )

        print("="*80)

        # ============================================================
        # SAVE RESULTS JSON
        # ============================================================
        final_results = {
            "test_metrics": test_results,

            "reproducibility_metrics": {
                "training_wall_clock_hours":
                    repro_metrics["training_time_hours"],

                "training_wall_clock_minutes":
                    repro_metrics["training_time_min"],

                "optimizer_steps":
                    total_steps,

                "mean_step_ms":
                    repro_metrics["mean_step_ms"],

                "p99_step_ms":
                    repro_metrics["p99_step_ms"],

                "peak_gpu_memory_allocated_gb":
                    repro_metrics["peak_gpu_memory_allocated_gb"],

                "peak_gpu_memory_reserved_gb":
                    repro_metrics["peak_gpu_memory_reserved_gb"],
            }
        }

        output_file = "LoRA_gemma4_seed42_VQA.json"

        with open(output_file, "w") as f:
            json.dump(final_results, f, indent=2)

        print(f"\nResults saved to: {output_file}")

        # ============================================================
        # SAVE FINAL MODEL
        # ============================================================
        print("\n" + "="*80)
        print("Saving final model...")

        output_dir = "./final_vqa_lora_only_gemma3_model"

        model.save_pretrained(output_dir)
        processor.save_pretrained(output_dir)

        print(f"Model saved to {output_dir}")

        # ============================================================
        # DONE
        # ============================================================
        print("\n" + "="*80)
        print("TRAINING COMPLETED SUCCESSFULLY")
        print("="*80 + "\n")

    except KeyboardInterrupt:
        print("\n" + "="*80)
        print("TRAINING INTERRUPTED BY USER")
        print("="*80)
        print("Saving checkpoint...")
        try:
            model.save_pretrained("./interrupted_checkpoint_lora_only_gemma3")
            processor.save_pretrained("./interrupted_checkpoint_lora_only_gemma3")
            print("Checkpoint saved to ./interrupted_checkpoint_lora_only_gemma3")
        except Exception as e:
            print(f"Failed to save checkpoint: {e}")

    except Exception as e:
        print("\n" + "="*80)
        print("TRAINING FAILED")
        print("="*80)
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
        raise

    finally:
        print("\nCleaning up...")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("Cleanup complete")


if __name__ == "__main__":
    main()

  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-84.0.0-py3-none-any.whl (818 kB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-81.0.0-py3-none-any.whl (1.1 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 84.0.0
    Uninstalling setuptools-84.0.0:
      Successfully uninstalled setuptools-84.0.0
Loading processor and model for Gemma 3-4B-IT...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Instrumented 81 vision modules + 102 language modules
Total LoRA modules: 183
Trainable params: 4,696,064
Total params: 4,304,775,536
Percentage: 0.109%
Loading VQAv2 dataset with splits...
Streaming VQAv2 dataset (no full download)...


Repo card metadata block was not found. Setting CardData to empty.


... (23 outputs left out)

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[Accuracy Debug] buffered answer-sets: 4 | predictions: 4
[Accuracy Debug] Sample 1
  Prediction (normalized): 'rail tracks'
  Raw human answers       : ['railroad tracks', 'track', 'railroad tracks', 'tracks', 'tracks', 'train tracks', 'track', 'tracks', 'rails', 'tracks']
  Normalized human answers: ['railroad tracks', 'track', 'railroad tracks', 'tracks', 'tracks', 'train tracks', 'track', 'tracks', 'rails', 'tracks']
[Accuracy Debug] Sample 2
  Prediction (normalized): 'using kites in open space provides ample room for them to fly and prevents them from hitting obstacles or people'
  Raw human answers       : ['it can get stock anywhere else', 'not get tangled in anything', 'open area to run', 'more airflow', "so they don't get tangled", 'they fly', "so they don't get caught up on its surroundings", 'wind', 'they need air', 'kites']
  Normalized human answers: ['it can get stock anywhere else', 'not get tangled in anything', 'open area to run', 'more airflow', "so they don't get t

Training Loss,Validation Loss,Step,Rouge L,Meteor,Bertscore F1,Moverscore,Accuracy
31.557268,21.017981,96,0.666667,0.484007,0.729361,0.730750,0.500000


Test Loss          : 21.01798
Test ROUGE-L       : 0.6667
Test METEOR         : 0.4840
Test BERTscore F1   : 0.7294
Test MoverScore     : 0.7307
Test Accuracy       : 0.5000

TRAINING REPRODUCIBILITY METRICS
Training wall-clock time : 0.0361 h
Training wall-clock time : 2.16 min
Optimizer steps          : 96
Mean step time           : 938.87 ms
P99 step time            : 1190.66 ms
Peak GPU memory allocated: 19.87 GB
Peak GPU memory reserved : 20.55 GB

Results saved to: LoRA_gemma4_seed42_VQA.json

Saving final model...
Model saved to ./final_vqa_lora_only_gemma3_model

TRAINING COMPLETED SUCCESSFULLY


Cleaning up...
Cleanup complete
